# eml-cost ecosystem demo

This notebook walks through the three packages that make up the
eml-cost ecosystem:

- **`eml-cost`** — Pfaffian profile of arbitrary symbolic
  expressions.
- **`eml-rewrite`** — F-family fusion-pattern rewriter with
  three-layer non-regression (cost · domain · numerical).
- **`eml-cost-torch`** — per-layer Pfaffian profile of any
  `torch.nn.Module`.

Six cells; runs end-to-end in under a minute.

## 1. Install

In [ ]:
# Run once. The torch package is optional — omit it if you only
# care about symbolic analysis.
%pip install --quiet eml-cost eml-rewrite eml-cost-torch torch

## 2. Pfaffian profile of 10 famous equations

Headline columns: `pfaffian_r` (total chain order),
`max_path_r` (deepest single-path chain order),
`predicted_depth` (the cost the rewriter / SymPy `simplify`
compares against), and a flag for the Pfaffian-but-not-EML
extension class.

In [ ]:
import sympy as sp
from eml_cost import analyze

x, mu, sigma, t, S, K, r, T, vol = sp.symbols(
    'x mu sigma t S K r T vol', positive=True
)

famous = {
    'quadratic root':      (-1 + sp.sqrt(1 + 4 * x)) / 2,
    'Gaussian pdf':        sp.exp(-((x - mu) ** 2) / (2 * sigma ** 2))
                            / (sigma * sp.sqrt(2 * sp.pi)),
    'Boltzmann factor':    sp.exp(-x / t),
    'sigmoid':             1 / (1 + sp.exp(-x)),
    'softplus':            sp.log(1 + sp.exp(x)),
    'tanh':                (sp.exp(x) - sp.exp(-x)) / (sp.exp(x) + sp.exp(-x)),
    'GELU (erf form)':     0.5 * x * (1 + sp.erf(x / sp.sqrt(2))),
    'Black-Scholes d1':    (sp.log(S / K) + (r + vol ** 2 / 2) * T)
                            / (vol * sp.sqrt(T)),
    'Schrodinger plane':   sp.exp(sp.I * x * t),
    'Planck spectrum':     1 / (sp.exp(x / t) - 1),
}

print(f"{'name':<24} {'r':>3} {'maxp':>4} {'pred':>4}  not-EML")
for name, expr in famous.items():
    a = analyze(expr)
    print(f"{name:<24} {a.pfaffian_r:>3} {a.max_path_r:>4} "
          f"{a.predicted_depth:>4}  {'*' if a.is_pfaffian_not_eml else ' '}")

## 3. Find and fix disguised expressions

`eml-rewrite` proposes equivalent rewrites that strictly reduce
the predicted EML cost, gated by the assumption system + a
30-digit numerical equivalence sweep. Below: 5 disguised forms
and what the rewriter prefers.

In [ ]:
from eml_rewrite import best, score

x = sp.Symbol('x', real=True, positive=True)

disguised = [
    sp.exp(x) / (1 + sp.exp(x)),                 # sigmoid (textbook form)
    sp.sinh(x) / sp.cosh(x),                     # tanh, unfused
    sp.sin(x) ** 2 + sp.cos(x) ** 2,             # Pythagorean = 1
    sp.cosh(x) ** 2 - sp.sinh(x) ** 2,           # hyperbolic id = 1
    (sp.exp(x) + sp.exp(-x)) / 2,                # cosh, unfused
]

print(f"{'before':<35} {'after':<25} {'Δ cost':>7}")
for expr in disguised:
    rewritten = best(expr)
    delta = score(expr) - score(rewritten)
    print(f"{str(expr):<35} {str(rewritten):<25} {delta:>+7}")

## 4. Per-layer Pfaffian profile of a PyTorch model

`eml-cost-torch` walks a `torch.nn.Module`, classifies each
layer's activation function, and emits a per-layer table similar
in shape to `torchsummary` but with Pfaffian columns.

In [ ]:
import torch.nn as nn
from eml_cost_torch import summary, profile

model = nn.Sequential(
    nn.Linear(64, 32), nn.ReLU(),       # r=0 layers
    nn.Linear(32, 16), nn.Sigmoid(),    # r=1
    nn.Linear(16, 8),  nn.Mish(),       # r=3 (deep activation)
    nn.Linear(8, 4),   nn.GELU(),       # non-EML (erf-based)
    nn.Linear(4, 2),
)
summary(model)

p = profile(model)
print(f"\nDerived: transcendental_ratio = {p.transcendental_ratio:.0%},"
      f" max_per_layer_r = {p.max_per_layer_r}")

## 5. Compare `eml_cost.measure` vs `count_ops` on disagreement cases

`count_ops` weights every operator equally. The `eml-cost`
measure tracks chain depth and parallel composition. They
disagree on five canonical cases.

In [ ]:
from sympy import count_ops
from eml_cost import measure as eml_measure

x, y = sp.symbols('x y', real=True)
cases = [
    sp.exp(sp.exp(sp.exp(x))),                 # deep composition
    sp.sin(x) * sp.cos(y),                     # independent vars
    1 / (1 + sp.exp(-x)),                      # canonical sigmoid
    sp.tanh(x),                                # fused vs unfused below
    sp.sinh(x) / sp.cosh(x),                   # mathematically = tanh(x)
]

print(f"{'expression':<28} {'count_ops':>10} {'eml_measure':>12}")
for expr in cases:
    print(f"{str(expr):<28} {count_ops(expr):>10} {eml_measure(expr):>12}")

## 6. Extension primitives — Bessel, Airy, Lambert W

These are Pfaffian (admit a polynomial-coefficient ODE chain)
but lie outside the strict EML-elementary class. The detector
flags them with `is_pfaffian_not_eml` and counts them via the
extension registry.

In [ ]:
from eml_cost import is_pfaffian_not_eml, PFAFFIAN_NOT_EML_R

x = sp.Symbol('x', real=True)
specials = [
    sp.besselj(0, x),
    sp.bessely(1, x),
    sp.airyai(x),
    sp.LambertW(x),
    sp.exp(x),                # in the EML class — NOT flagged
    sp.sin(x),                # in the EML class — NOT flagged
]

print(f"{'expression':<22} {'flagged?':<10} {'registry r':>10}")
for expr in specials:
    name = type(expr).__name__
    flagged = is_pfaffian_not_eml(expr)
    reg_r = PFAFFIAN_NOT_EML_R.get(name, '—')
    print(f"{str(expr):<22} {str(flagged):<10} {reg_r:>10}")

---

**Where to next.**

- `pip install eml-rewrite` and try the `eml-rewrite scan FILE`
  CLI on a real codebase.
- Use `eml_cost.measure` as a `simplify(measure=...)` argument
  in your own SymPy work.
- For NN profiling, `summary(model)` is the one-line interface;
  the underlying `profile(model)` returns a structured
  `ModelProfile` if you want to programmatically post-process.